In [5]:
from pathlib import Path
import pandas as pd
import re
from datetime import datetime

# ============================================
# 0) 기본 설정
# ============================================
BASE_DIR = Path(r"C:\Users\user\Desktop\RAW_LOG")  # FCT 로그 최상위 폴더
TC_FOLDERS = ["TC6", "TC7", "TC8", "TC9"]          # 중간 폴더
TARGET_FOLDERS = ["GoodFile", "BadFile"]           # 하위 폴더

# 이력 CSV 경로
HISTORY_CSV_PATH = Path(r"C:\Users\user\Desktop\a4_fct_json_table_processing_history.csv")

# ============================================
# 1) 이력 로드 & 저장
# ============================================
def load_history(path: Path) -> pd.DataFrame:
    if path.exists():
        try:
            hist_df = pd.read_csv(path)
            if "file_path" not in hist_df.columns:
                hist_df = pd.DataFrame(columns=["file_path", "processed_time"])
        except Exception:
            hist_df = pd.DataFrame(columns=["file_path", "processed_time"])
    else:
        hist_df = pd.DataFrame(columns=["file_path", "processed_time"])
    return hist_df


def save_history(path: Path, hist_df: pd.DataFrame):
    if not hist_df.empty:
        hist_df = hist_df.sort_values("processed_time").drop_duplicates("file_path", keep="last")
    hist_df.to_csv(path, index=False, encoding="utf-8-sig")


# ============================================
# 2) 파싱용 정규식
# ============================================
STATION_PATTERN = re.compile(r"Station\s*:?\s*(\S+)", re.IGNORECASE)
BARCODE_PATTERN = re.compile(r"Barcode\s+information\s*:?\s*(.+)", re.IGNORECASE)
STEP_PATTERN = re.compile(
    r"^(?P<desc>.+?)\s*,\s*(?P<value>[^,]*),\s*(?P<min>[^,]*),\s*(?P<max>[^,]*),\s*(?P<result>\[[^\]]*\])"
)

def normalize_step_desc(desc: str) -> str:
    # 2개 이상 공백 -> 1개, 양끝 공백 제거
    return " ".join(desc.split())


def parse_fct_file(file_path: Path):
    """한 개 파일을 파싱해서 레코드 리스트 반환"""
    try:
        with file_path.open("r", encoding="cp949", errors="ignore") as f:
            lines = [line.rstrip("\n") for line in f]
    except UnicodeDecodeError:
        with file_path.open("r", encoding="utf-8", errors="ignore") as f:
            lines = [line.rstrip("\n") for line in f]

    if not lines:
        return []

    station = None
    barcode = None

    # Station (3번째 줄 우선)
    if len(lines) >= 3:
        m = STATION_PATTERN.search(lines[2])
        if m:
            station = m.group(1).strip()
    if station is None:
        for line in lines:
            m = STATION_PATTERN.search(line)
            if m:
                station = m.group(1).strip()
                break

    # Barcode (5번째 줄 우선)
    if len(lines) >= 5:
        m = BARCODE_PATTERN.search(lines[4])
        if m:
            barcode = m.group(1).strip()
    if barcode is None:
        for line in lines:
            m = BARCODE_PATTERN.search(line)
            if m:
                barcode = m.group(1).strip()
                break

    records = []
    for line in lines:
        m = STEP_PATTERN.match(line)
        if not m:
            continue

        desc_raw = m.group("desc")
        value_raw = m.group("value")
        min_raw = m.group("min")
        max_raw = m.group("max")
        result_raw = m.group("result")

        step_desc = normalize_step_desc(desc_raw)

        records.append({
            "file_path": str(file_path),
            "Station": station if station is not None else "",
            "barcode": barcode if barcode is not None else "",
            "step_description": step_desc,
            "value": str(value_raw).strip(),
            "min": str(min_raw).strip(),
            "max": str(max_raw).strip(),
            "result": str(result_raw).strip(),
        })

    return records


# ============================================
# 3) 폴더 스캔 & 메인 실행
# ============================================
def collect_fct_files(base_dir: Path) -> list[Path]:
    file_list = []
    for tc in TC_FOLDERS:
        tc_path = base_dir / tc
        if not tc_path.exists():
            continue

        for date_dir in tc_path.iterdir():
            if not date_dir.is_dir():
                continue

            for gb in TARGET_FOLDERS:
                target_dir = date_dir / gb
                if not target_dir.exists():
                    continue

                for f in target_dir.glob("*.txt"):
                    file_list.append(f)
    return file_list


def run_fct_parsing():
    history_df = load_history(HISTORY_CSV_PATH)
    processed_files = set(history_df["file_path"].tolist()) if not history_df.empty else set()

    all_files = collect_fct_files(BASE_DIR)
    new_files = [f for f in all_files if str(f) not in processed_files]

    print(f"총 파일 수: {len(all_files)}개")
    print(f"이미 처리된 파일 수(이력): {len(processed_files)}개")
    print(f"이번에 새로 처리할 파일 수: {len(new_files)}개")

    all_records = []

    for idx, f in enumerate(new_files, start=1):
        recs = parse_fct_file(f)
        all_records.extend(recs)

        if idx % 1000 == 0 or idx == len(new_files):
            print(f"  → 현재 {idx}/{len(new_files)} 파일 처리 완료")

    if all_records:
        df = pd.DataFrame(all_records)
        # file_path 제거 + 컬럼 순서 지정
        cols = ["Station", "barcode", "step_description", "value", "min", "max", "result"]
        df = df[cols].reset_index(drop=True)   # ← RangeIndex 0,1,2,... 로 맞춤
    else:
        df = pd.DataFrame(
            columns=["Station", "barcode", "step_description", "value", "min", "max", "result"]
        )

    # 이력 업데이트
    if new_files:
        now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        new_hist_rows = [{"file_path": str(f), "processed_time": now_str} for f in new_files]
        new_hist_df = pd.DataFrame(new_hist_rows)

        if history_df.empty:
            updated_hist = new_hist_df
        else:
            updated_hist = pd.concat([history_df, new_hist_df], ignore_index=True)

        save_history(HISTORY_CSV_PATH, updated_hist)

    return df


# ============================================
# 4) 실행
# ============================================
fct_table = run_fct_parsing()
fct_table

총 파일 수: 14987개
이미 처리된 파일 수(이력): 0개
이번에 새로 처리할 파일 수: 14987개
  → 현재 1000/14987 파일 처리 완료
  → 현재 2000/14987 파일 처리 완료
  → 현재 3000/14987 파일 처리 완료
  → 현재 4000/14987 파일 처리 완료
  → 현재 5000/14987 파일 처리 완료
  → 현재 6000/14987 파일 처리 완료
  → 현재 7000/14987 파일 처리 완료
  → 현재 8000/14987 파일 처리 완료
  → 현재 9000/14987 파일 처리 완료
  → 현재 10000/14987 파일 처리 완료
  → 현재 11000/14987 파일 처리 완료
  → 현재 12000/14987 파일 처리 완료
  → 현재 13000/14987 파일 처리 완료
  → 현재 14000/14987 파일 처리 완료
  → 현재 14987/14987 파일 처리 완료


,Station,barcode,step_description,value,min,max,result
0,FCT1,BA1WJ25273503681SJ8T-14F014-AE,1.01 Test Input Voltage(V),14.67,14.60,14.80,[PASS]
1,FCT1,BA1WJ25273503681SJ8T-14F014-AE,1.02_Test_USB2_error(Type-C_A_side),0,0,0,[PASS]
2,FCT1,BA1WJ25273503681SJ8T-14F014-AE,1.03_Test USB2 benchmark.maxrd(Mbit/s),360Mbit/s,330,400,[PASS]
3,FCT1,BA1WJ25273503681SJ8T-14F014-AE,1.04 Test USB2 benchmark.maxwr(Mbit/s),360Mbit/s,330,400,[PASS]
4,FCT1,BA1WJ25273503681SJ8T-14F014-AE,1.05 Test USB2 benchmark.avgrw(Mbit/s),343Mbit/s,330,400,[PASS]
...,...,...,...,...,...,...,...
527209,FCT4,BA1WJ25276500236SJ8T-14F014-AE,1.32_PD Negotiation SET PDO4,,,,[PASS]
527210,FCT4,BA1WJ25276500236SJ8T-14F014-AE,1.33_Test_VUSB_Type-C_A(ELoad2=1.35A)cur,1.295,1.25,1.45,[PASS]
527211,FCT4,BA1WJ25276500236SJ8T-14F014-AE,1.34_Test_VUSB_Type-C_A(ELoad2=1.35A)vol,20.138,19.5,21,[PASS]
527212,FCT4,BA1WJ25276500236SJ8T-14F014-AE,1.35 Test Check CC1 level(A side),1.96,1.8,3,[PASS]
